# The wind inside a storm at one time: arrows, cutaways and streamlines

A rendered cloud hides the air motion inside it. This notebook shows the
3-D wind of one RAMS output time in four ways:

1. arrows on a vertical plane through the updraft, inside a thinned cloud;
2. the same arrows on the face of a *cutaway*, a cloud cut open along a plane;
3. storm-relative streamlines through the updraft;
4. the same streamlines inside a cutaway cloud.

It ends with a labelled side-by-side figure built with `sv.compose`.

Every panel uses one camera, so the panels line up. Section 2 shows how.

Run the notebook in the pixi `blender` environment, after
`01_storm_portrait.ipynb` explains the basics. Paths are in
`notebook_setup.py`.

In [ ]:
import sys
from pathlib import Path

# notebook_setup.py sits next to this notebook. Jupyter starts the kernel
# either in this folder or in the repository root (VS Code's default).
for notebook_folder in (Path.cwd(), Path.cwd() / "examples" / "notebooks"):
    if (notebook_folder / "notebook_setup.py").exists():
        sys.path.insert(0, str(notebook_folder))
        break

import numpy as np
import xarrams

import notebook_setup as setup
import skyvista as sv

## 1. Load the cloud and the wind

`xarrams.read_rams_output` reads the hydrometeors, as in notebook 01, and
the wind components `UC`, `VC` and `WC` (m/s).

RAMS stores each wind component on the faces of its grid cells. This
layout is a *staggered* grid. `UC[i]` sits on the east face of cell `i`,
`VC[j]` on its north face and `WC[k]` on its top face. `xarrams` labels
them with the cell-center coordinates all the same.

`center_winds` below averages each component from the two faces of a cell
onto its center, where the hydrometeors live. Every variable then shares
one set of coordinates. The vertical average needs the face below the
lowest kept level, so it runs before the levels are selected.

In [ ]:
# RAMS hydrometeor mixing ratios (R?P), number concentrations (C?P), the
# base-state air density, and the staggered wind components.
RAMS_CLOUD_VARIABLES = [
    "RCP", "RDP", "RRP", "RPP", "RSP", "RAP", "RGP", "RHP",
    "CCP", "CDP", "CRP", "CPP", "CSP", "CAP", "CGP", "CHP",
    "DN0",
]
RAMS_WIND_VARIABLES = ["UC", "VC", "WC"]
# Levels kept: from the ground to the highest cloud (m).
LOWEST_HEIGHT, TOP_HEIGHT = 0.0, 16500.0
# RAMS output names count model time from this date.
RUN_START = np.datetime64("2000-01-01T00:00:00")


def center_winds(rams_output):
    """
    U, V and W at the cell centers, from RAMS's face-centered UC, VC, WC.

    The center of cell i lies between its lower face (stored at i - 1)
    and its upper face (stored at i), so its value is their mean.
    ``shift(... = 1)`` lines each lower face up with its cell. The first
    cell has no lower face in the array, so it takes its upper face.
    """
    centered = {}
    for staggered_name, centered_name, axis in (("UC", "U", "x"), ("VC", "V", "y"), ("WC", "W", "z")):
        upper_faces = rams_output[staggered_name]
        lower_faces = upper_faces.shift({axis: 1}).fillna(upper_faces)
        centered[centered_name] = 0.5 * (lower_faces + upper_faces)
    return rams_output.drop_vars(RAMS_WIND_VARIABLES).assign(centered)


# One output time of an isolated storm (94 min), winds centered first.
storm_output = xarrams.read_rams_output(
    [setup.RAMS_STORM_FILE], keep_vars=RAMS_CLOUD_VARIABLES + RAMS_WIND_VARIABLES, silent=True
)
storm_output = center_winds(storm_output.isel(time=0))
storm_output = storm_output.sel(z=slice(LOWEST_HEIGHT, TOP_HEIGHT)).load()

# The cloud and the wind, as two datasets on the same grid.
storm_clouds = storm_output.drop_vars(["U", "V", "W"])
storm_winds = storm_output[["U", "V", "W"]]
storm_minutes = (storm_output["time"].values - RUN_START) / np.timedelta64(1, "m")
print(f"t = {storm_minutes:.1f} min; w from {float(storm_winds['W'].min()):.1f}"
      f" to {float(storm_winds['W'].max()):.1f} m/s")

## 2. One camera for every panel

The automatic camera frames the bounding box of everything exported.
Arrows and streamlines extend beyond the cloud, so each panel would get a
different view.

The fix takes two steps:

1. Render the plain cloud with the automatic camera.
2. Read that camera back from the bundle with `sv.BlenderView.from_bundle`.
   `config.use_view(view)` then sets it in every later panel's config.

A view also holds the *origin shift*: the model point that becomes
Blender's origin. Every panel must use the same one, or the same camera
coordinates would point at different places.

Every panel starts from the same settings, made by `make_panel_config`:

- preview quality, at 800 x 600 per panel;
- the camera and sun of notebook 01, section 3.

In [ ]:
# Panels are 4:3, so two side by side make a wide figure.
PANEL_RESOLUTION = (800, 600)


def make_panel_config():
    """Physical-cloud settings at preview quality, with notebook 01's framing."""
    config = sv.BlenderExportConfig.for_clouds(quality="preview")
    config.render.resolution = PANEL_RESOLUTION
    # A low camera south-east of the storm, looking slightly up.
    config.camera.azimuth_deg = -55.0
    config.camera.elevation_deg = -8.0
    config.camera.distance_factor = 1.1
    # The sun 30 degrees up, from the north-east.
    config.world.sun_azimuth_deg = 15.0
    config.world.sun_elevation_deg = 30.0
    return config


# Step 1: the plain cloud, with the automatic camera.
plain_bundle_dir = setup.WORK_DIR / "winds_plain_cloud"
sv.Scene().add_clouds(storm_clouds, microphysics="rams").to_blender(
    plain_bundle_dir, config=make_panel_config(), render=True, quiet=True
)
setup.show_image(plain_bundle_dir, width=600)

# Step 2: its camera and origin shift, for every later panel.
plain_cloud_view = sv.BlenderView.from_bundle(plain_bundle_dir)
print(plain_cloud_view)


def make_pinned_config():
    """Panel settings with the plain cloud's camera and origin."""
    config = make_panel_config().use_view(plain_cloud_view)
    # The composed figure draws its own colorbar (section 7).
    config.overlay_colorbars = False
    return config

## 3. Arrows on a vertical plane, inside a thinned cloud

The arrows sit on one vertical plane (one `y` row of the grid) through the
strongest updraft. Each arrow shows the full 3-D wind at its point, so it
can point out of the plane.

- `stride` keeps every 12th grid point along `x` and `z`.
- `factor` sets the arrow length per unit of wind speed (here 80 m per m/s).
- `glyph_width` fixes the shaft width (m), so weak and strong arrows are
  equally easy to see.
- `color_by="w"` colors each arrow by its vertical velocity.
- `mask_by` names a boolean variable; arrows are drawn only where it is true.

The mask drops arrows that carry little information: weak vertical motion
(|w| < 2 m/s) in clear air. Arrows in cloud are always kept.

A full-strength cloud would hide the arrows. `extinction_scale=0.03` keeps
3% of the cloud's extinction, so the arrows inside it show. The `self_lit`
shader makes each arrow emit its own color. Its color then reads in the
cloud's shadow too.

In [ ]:
# The plane: the y row through the strongest updraft (kept as a length-1 dim).
strongest_updraft_index = np.unravel_index(
    int(np.nanargmax(storm_winds["W"].values)), storm_winds["W"].shape
)
plane_y_index = int(strongest_updraft_index[storm_winds["W"].dims.index("y")])
wind_plane = storm_winds.isel(y=[plane_y_index])
plane_y = float(wind_plane["y"].values[0])

# Clutter filter: strong vertical motion, or a cloudy point.
mixing_ratio_names = [name for name in storm_clouds.data_vars if name.startswith("R")]
total_mixing_ratio = sum(storm_clouds[name] for name in mixing_ratio_names)
cloud_on_plane = total_mixing_ratio.isel(y=[plane_y_index]) > 1.0e-5
strong_vertical_motion = np.abs(wind_plane["W"]) >= 2.0
wind_plane["show_arrow"] = strong_vertical_motion | cloud_on_plane

print(f"plane at y = {plane_y / 1e3:.2f} km;"
      f" {int(wind_plane['show_arrow'].sum())} of {wind_plane['show_arrow'].size} points kept")

In [ ]:
# Vertical velocity at the ends of the arrow colormap (m/s).
W_LIMIT = 6.0


def make_wind_arrows(colormap):
    """Arrows of the full 3-D wind on the plane, colored by w."""
    arrows = sv.make_vectors(
        "wind",
        u="U", v="V", w="W",
        stride={"x": 12, "z": 12},
        color_by="w",
        cmap=colormap,
        clim=(-W_LIMIT, W_LIMIT),
        factor=80.0,
        glyph_width=65.0,
        mask_by="show_arrow",
    )
    # Each arrow emits its own color, so it reads in shadow.
    arrows.appearance.shader = "self_lit"
    return arrows


thinned_arrows_bundle_dir = setup.WORK_DIR / "winds_arrows_thinned"
thinned_arrows_scene = sv.Scene().add_clouds(
    storm_clouds, microphysics="rams", extinction_scale=0.03
)
thinned_arrows_scene.add(wind_plane, make_wind_arrows("RdBu_r"))
thinned_arrows_scene.to_blender(
    thinned_arrows_bundle_dir, config=make_pinned_config(), render=True, quiet=True
)
setup.show_image(thinned_arrows_bundle_dir, width=600)

## 4. Arrows on a cutaway

A cutaway removes the cloud on one side of a plane and keeps the rest at
full strength. `cutaway=(point, normal)` removes the cloud on the side the
normal points to. Here the normal is -y, toward the camera, so the
camera looks at the cut face.

Two details keep the arrows readable:

- The arrows move 300 m toward the camera. An arrow whose wind points into
  the cloud would otherwise vanish behind the cut face.
- The cut face is bright white, so white arrows (w near 0) would disappear.
  This colormap runs blue, grey, red instead. The colormap is registered
  with matplotlib by name, because the exporter looks colormaps up by name.

In [ ]:
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap

# Blue -> grey -> red: the end colors of RdBu, with grey in the middle.
CUTAWAY_COLORMAP = "skyvista_blue_grey_red"
if CUTAWAY_COLORMAP not in colormaps:
    colormaps.register(
        LinearSegmentedColormap.from_list(
            CUTAWAY_COLORMAP, ("#2166ac", "#808080", "#b2182b")
        )
    )

# Arrows 300 m in front of the cut face.
ARROW_OFFSET_TOWARD_CAMERA = 300.0
offset_wind_plane = wind_plane.assign_coords(
    y=wind_plane["y"] - ARROW_OFFSET_TOWARD_CAMERA
)

# Remove the cloud on the camera side (y below the plane).
cutaway_plane = ((0.0, plane_y, 0.0), (0.0, -1.0, 0.0))

cutaway_arrows_bundle_dir = setup.WORK_DIR / "winds_arrows_cutaway"
cutaway_arrows_scene = sv.Scene().add_clouds(
    storm_clouds, microphysics="rams", cutaway=cutaway_plane
)
cutaway_arrows_scene.add(offset_wind_plane, make_wind_arrows(CUTAWAY_COLORMAP))
cutaway_arrows_scene.to_blender(
    cutaway_arrows_bundle_dir, config=make_pinned_config(), render=True, quiet=True
)
setup.show_image(cutaway_arrows_bundle_dir, width=600)

## 5. Storm-relative streamlines

A *streamline* is a curve that is everywhere tangent to the wind at one
instant. It shows the pattern of the flow at that time. It is not the path
that an air parcel follows over time.

A streamline's shape depends on the frame of reference. In the ground's
frame, the environmental wind sweeps every line downstream and hides the
storm's own circulation. `frame_velocity` subtracts a velocity before
tracing. Here it is the mean wind of the lowest 6 km, a common estimate
of storm motion.

The other settings:

- `seed_mask_by` and `seed_stride`: seeds sit on every 24th grid point, and
  only inside the updraft (w > 5 m/s).
- `max_length`: each streamline runs up to 25 km each way from its seed.
- `arrow_spacing`: a cone every 2.5 km along each tube points downstream.
- `point_spacing`: the traced line keeps one point every 100 m.

skyvista traces the streamlines in Python with VTK. Blender then sweeps a
tube along each one and places the cones.

In [ ]:
# Storm motion: the mean horizontal wind between the ground and 6 km.
STORM_MOTION_LAYER_TOP = 6000.0
storm_motion_layer = storm_winds.sel(z=slice(0.0, STORM_MOTION_LAYER_TOP))
storm_motion = (
    float(storm_motion_layer["U"].mean()),
    float(storm_motion_layer["V"].mean()),
    0.0,
)
print(f"storm motion ({storm_motion[0]:.1f}, {storm_motion[1]:.1f}) m/s")

# Seeds only inside the updraft.
storm_winds["in_updraft"] = storm_winds["W"] > 5.0

# Vertical velocity at the ends of the streamline colormap (m/s).
STREAMLINE_W_LIMIT = 12.0
updraft_streamlines = sv.make_streamlines(
    u="U", v="V", w="W",
    seed_stride=24,
    seed_mask_by="in_updraft",
    max_length=25000.0,
    min_length=2000.0,
    point_spacing=100.0,
    frame_velocity=storm_motion,
    color_by="w",
    cmap="RdBu_r",
    clim=(-STREAMLINE_W_LIMIT, STREAMLINE_W_LIMIT),
    tube_radius=45.0,
    arrow_spacing=2500.0,
    shader="self_lit",
)

thinned_streamlines_bundle_dir = setup.WORK_DIR / "winds_streamlines_thinned"
thinned_streamlines_scene = sv.Scene().add_clouds(
    storm_clouds, microphysics="rams", extinction_scale=0.03
)
thinned_streamlines_scene.add(storm_winds, updraft_streamlines)
thinned_streamlines_scene.to_blender(
    thinned_streamlines_bundle_dir, config=make_pinned_config(), render=True, quiet=True
)
setup.show_image(thinned_streamlines_bundle_dir, width=600)

## 6. Streamlines inside a cutaway

The thinned cloud shows every streamline but hardly any cloud. A cutaway
keeps the far half of the cloud at full strength, as a backdrop. The
streamlines on the camera side of the cut stay in front of it.

This scene reuses the streamline spec and the cut plane of section 4.

In [ ]:
cutaway_streamlines_bundle_dir = setup.WORK_DIR / "winds_streamlines_cutaway"
cutaway_streamlines_scene = sv.Scene().add_clouds(
    storm_clouds, microphysics="rams", cutaway=cutaway_plane
)
cutaway_streamlines_scene.add(storm_winds, updraft_streamlines)
cutaway_streamlines_scene.to_blender(
    cutaway_streamlines_bundle_dir, config=make_pinned_config(), render=True, quiet=True
)
setup.show_image(cutaway_streamlines_bundle_dir, width=600)

## 7. A labelled side-by-side figure

`sv.compose` lays renders out with matplotlib, on a canvas exactly the
size of the images. Its pieces:

- `Figure`: a grid of panels with a gap between them.
- `Panel`: one image and the overlays drawn on it.
- `Text`: a label at an anchor ("upper left", "lower right", ...).
- `Colorbar`: a colorbar drawn from a matplotlib colormap and limits.

Sizes and offsets are fractions of the panel height, so a layout looks the
same at any resolution. Text is white with a black outline, which reads on
sky, cloud and ground.

The colorbar here uses the same colormap and limits as the streamlines, so
a color on a tube reads directly off the bar.

In [ ]:
# Text heights, as fractions of the panel height.
TITLE_SIZE, LABEL_SIZE = 0.04, 0.03

left_panel = sv.compose.Panel(
    setup.rendered_frame(plain_bundle_dir),
    [
        sv.compose.Text("Cloud, physically based", size=TITLE_SIZE),
        sv.compose.Text(
            f"t = {storm_minutes:.1f} min", size=LABEL_SIZE, offset=(0.0, -0.055)
        ),
    ],
)
right_panel = sv.compose.Panel(
    setup.rendered_frame(cutaway_streamlines_bundle_dir),
    [
        sv.compose.Text("Storm-relative streamlines, cloud cut open", size=TITLE_SIZE),
        sv.compose.Text(
            f"Cut at y = {plane_y / 1e3:.1f} km\nCones point downstream",
            anchor="lower left",
            size=LABEL_SIZE,
        ),
        sv.compose.Colorbar(
            cmap="RdBu_r",
            clim=(-STREAMLINE_W_LIMIT, STREAMLINE_W_LIMIT),
            label="Vertical velocity w (m/s)",
            ticks=list(np.linspace(-STREAMLINE_W_LIMIT, STREAMLINE_W_LIMIT, 5)),
            anchor="lower right",
            length=0.35,
            text_size=LABEL_SIZE,
        ),
    ],
)
figure_path = setup.WORK_DIR / "winds_side_by_side.png"
sv.compose.Figure(panels=[left_panel, right_panel]).render(figure_path)
setup.show_image(figure_path, width=1000)

## Where to go from here

- `examples/rams_clouds/render_rams_wind_side_by_side.py` makes the arrow
  figure from the command line. Its clutter filter keeps arrows within a
  set 3-D distance of cloud, rather than only arrows in cloud.
- `examples/rams_clouds/render_rams_streamlines_side_by_side.py` makes the
  streamline figure, with aerial perspective.
- `05_flow_at_a_fixed_time.ipynb` animates these streamlines: the storm stays
  fixed while the cones move along the tubes with the air.